# 05 - Loss Functions y Maximum Likelihood

**AI sin humo** - Notas personales para entender deep learning desde cero.

En el notebook 01 vimos MSE para regresión y cross-entropy para clasificación. Acá vamos a profundizar en **por qué** esas loss functions son las que son, usando el framework de **Maximum Likelihood**. Esto da una justificación formal y elegante.

---

## Contenido

1. [¿Qué es una loss function?](#que-es)
2. [Maximum Likelihood: la idea general](#maximum-likelihood)
3. [Negative Log-Likelihood](#nll)
4. [MSE sale de asumir distribución Gaussiana](#mse-gaussiana)
5. [Cross-entropy sale de asumir distribución categórica](#ce-categorica)
6. [Resumen de losses comunes](#resumen)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---

<a id='que-es'></a>
## 1. ¿Qué es una loss function?

Es una función que toma las predicciones del modelo y las compara con los valores reales (targets). Devuelve un número que indica **qué tan mal le fue**. Mientras más alto ese número, peor la predicción.

El entrenamiento consiste en **minimizar** esa función. Todo el sistema de backpropagation y gradient descent existe para este propósito: encontrar los parámetros que hagan que la loss sea lo más baja posible.

La pregunta es: **¿por qué usamos esas funciones de loss específicas y no cualquier otra?** La respuesta viene de la estadística.

---

<a id='maximum-likelihood'></a>
## 2. Maximum Likelihood: la idea general

Maximum Likelihood (máxima verosimilitud) es un principio estadístico que dice:

> **Elegí los parámetros que hagan que los datos observados sean los más probables posible.**

Es decir: dado que ya vimos estos datos, ¿qué parámetros del modelo harían que estos datos sean los más "esperables"?

Formalmente, si tenemos $N$ datos independientes, queremos maximizar:

$$\mathcal{L}(\phi) = \prod_{i=1}^{N} P(y_i | x_i, \phi)$$

Donde $P(y_i | x_i, \phi)$ es la probabilidad de observar $y_i$ dado $x_i$ y los parámetros $\phi$.

El producto es porque asumimos que los datos son **independientes**: la probabilidad conjunta de todos es el producto de las individuales.

---

<a id='nll'></a>
## 3. Negative Log-Likelihood

Multiplicar muchas probabilidades (números entre 0 y 1) da un número **astronómicamente chico**. Esto causa problemas numéricos (underflow).

El truco es usar el **logaritmo**. Como $\log(a \cdot b) = \log(a) + \log(b)$, transformamos el producto en suma:

$$\log \mathcal{L}(\phi) = \sum_{i=1}^{N} \log P(y_i | x_i, \phi)$$

Y como queremos **minimizar** (en vez de maximizar), usamos el **negativo**:

$$\text{NLL}(\phi) = -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i | x_i, \phi)$$

**Minimizar la NLL = maximizar el likelihood**. Son lo mismo pero en la dirección que nos conviene para gradient descent.

Ahora, dependiendo de qué distribución asumimos para $P(y|x,\phi)$, nos sale una loss function diferente.

---

<a id='mse-gaussiana'></a>
## 4. MSE sale de asumir distribución Gaussiana

Para **regresión**: si asumimos que $y$ se distribuye como una Gaussiana centrada en la predicción del modelo:

$$P(y | x, \phi) = \mathcal{N}(y; f(x, \phi), \sigma^2)$$

Donde $f(x, \phi)$ es la predicción del modelo (la media de la Gaussiana) y $\sigma^2$ es la varianza (que asumimos constante).

Si metemos esto en la NLL y simplificamos (sacamos constantes que no dependen de $\phi$):

$$\text{NLL} \propto \frac{1}{N} \sum_{i=1}^{N} (y_i - f(x_i, \phi))^2$$

¡Eso es **MSE**! El Mean Squared Error sale naturalmente de asumir que los errores se distribuyen de forma Gaussiana. No es arbitrario.

In [ ]:
# Visualize: Gaussian distribution centered at the model prediction
from scipy.stats import norm

y_pred = 3.0  # model prediction (mean of Gaussian)
sigma = 0.5
y_range = np.linspace(1, 5, 200)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(y_range, norm.pdf(y_range, y_pred, sigma), 'b-', lw=2, label=f'P(y|x) ~ N({y_pred}, {sigma}²)')

# Two observations
y_close = 3.2
y_far = 4.5
ax.axvline(y_close, color='green', linestyle='--', label=f'y={y_close} → alta prob, loss baja')
ax.axvline(y_far, color='red', linestyle='--', label=f'y={y_far} → baja prob, loss alta')

ax.set_xlabel('y (target)')
ax.set_ylabel('P(y | x, φ)')
ax.set_title('MSE = maximizar la probabilidad bajo distribución Gaussiana')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---

<a id='ce-categorica'></a>
## 5. Cross-entropy sale de asumir distribución categórica

Para **clasificación**: si tenemos $K$ clases, el modelo produce un vector de probabilidades $[p_1, p_2, ..., p_K]$ (vía softmax) y la clase verdadera es $t$.

La distribución es **categórica**:

$$P(y = t | x, \phi) = p_t$$

La NLL es:

$$\text{NLL} = -\frac{1}{N} \sum_{i=1}^{N} \log(p_{t_i})$$

Que es exactamente **cross-entropy**.

Si usamos representación one-hot ($y$ es un vector con un 1 en la posición correcta y 0 en el resto):

$$\text{CE} = -\sum_{k=1}^{K} y_k \log(p_k)$$

Como $y_k = 0$ para todas las clases excepto la correcta, solo queda $-\log(p_{\text{clase correcta}})$.

In [ ]:
def softmax(logits):
    exp_logits = np.exp(logits - np.max(logits))
    return exp_logits / exp_logits.sum()


# Example: 4-class classification
logits_good = np.array([3.0, 0.5, -1.0, 0.1])  # class 0 has highest logit
logits_bad = np.array([0.1, 0.5, -1.0, 3.0])    # class 3 has highest, but true is 0
target = 0

probs_good = softmax(logits_good)
probs_bad = softmax(logits_bad)

loss_good = -np.log(probs_good[target])
loss_bad = -np.log(probs_bad[target])

print("Modelo que acierta:")
print(f"  Probs: {np.round(probs_good, 3)} → P(clase {target}) = {probs_good[target]:.3f}")
print(f"  Cross-entropy loss: {loss_good:.3f} (baja)")

print("\nModelo que falla:")
print(f"  Probs: {np.round(probs_bad, 3)} → P(clase {target}) = {probs_bad[target]:.3f}")
print(f"  Cross-entropy loss: {loss_bad:.3f} (alta)")

---

<a id='resumen'></a>
## 6. Resumen de losses comunes

| Problema | Distribución asumida | Loss function | Fórmula |
|----------|---------------------|---------------|----------|
| **Regresión** | Gaussiana | **MSE** | $\frac{1}{N}\sum(y_i - \hat{y}_i)^2$ |
| **Clasificación binaria** | Bernoulli | **BCE** | $-[y\log(p) + (1-y)\log(1-p)]$ |
| **Clasificación multiclase** | Categórica | **Cross-Entropy** | $-\log(p_{\text{clase correcta}})$ |

**La clave**: las loss functions no son arbitrarias. Salen de asumir una distribución para los datos y aplicar el principio de Maximum Likelihood.

---

**Siguiente notebook →** [06 - Backpropagation](./06_backpropagation.ipynb): cómo se calculan los gradientes para entrenar la red.